# IMPORT DATASETS 

In [2]:
!pip install datasets

In [3]:
pip install -U datasets

Note: you may need to restart the kernel to use updated packages.


In [4]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

/Users/rhythemsabharwalgmail.com/Desktop/SLM/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Exploration

In [5]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [6]:
ds["train"].column_names

['text']

In [7]:
ds["train"].select(range(5)).to_pandas()

,text
0,"One day, a little girl named Lily found a need..."
1,"Once upon a time, there was a little car named..."
2,"One day, a little fish named Fin was swimming ..."
3,"Once upon a time, in a land full of trees, the..."
4,"Once upon a time, there was a little girl name..."


In [8]:
len(ds["train"])

2119719

In [9]:
lengths = [len(x["text"]) for x in ds["train"].select(range(1000))]

In [10]:
import numpy as np

print(np.mean(lengths))
print(np.max(lengths))
print(np.min(lengths))

941.64
4123
274


In [11]:
sum(x["text"] is None for x in ds["train"])

0

In [12]:
sum(len(x["text"]) == 0 for x in ds["train"])

230

# Tokenize the Dataset

## In this step, we will:

### (1) Convert the text into token IDs.
Language models cannot process raw text directly, so each story is converted into a sequence of numerical token IDs using a tokenizer.

### (2) Save the token IDs into `train.bin` and `validation.bin`.
Instead of storing the original text, we save the processed token IDs in binary files. This allows us to load the training data much faster during model training.

### (3) Store the processed data on disk.
Keeping the token IDs in binary files on disk avoids repeatedly tokenizing the dataset and reduces RAM usage, making training more efficient, especially for large datasets.

In [ ]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

# here we are using gpt2 tokenizer as gpt4 or gpt5 tokenizer would take more space there would'nt be much of a difference.
enc = tiktoken.get_encoding("gpt2")

def process(example):
    # Convert the text into token IDs without adding any special tokens.
    ids = enc.encode_ordinary(example['text'])
    # Add <|endoftext|> token (50256) at the end of each story so the model learns story boundaries.
    ids.append(enc.eot_token) # 50256
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map( #maps the stories one by one to the process function
        process,
        remove_columns=['text'], # since the stories are converted into token ids so text inside them is not needed as it will take extra space on ram
        desc="tokenizing the splits",
        num_proc=8, # uses all 8 cores of cpu to tokenization simultaneosly 
        )
    
    # Combine all token IDs into a single binary file for efficient training.
    for split, dset in tokenized.items(): # same code works for both datasets without writing it twice.
        arr_len = np.sum(dset['len'], dtype=np.uint64) # used to calculate total token ids nd stores the result as 64bit integer 
        filename = f'{split}.bin'

        dtype = np.uint16

        # Create a memory-mapped binary file to store the token IDs on disk.
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):

            # Process the dataset in batches for faster disk writes.
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')

            arr_batch = np.concatenate(batch['ids'])

            # Write the current batch of token IDs into the binary file.
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        # Ensure all data is written from memory to disk.
        arr.flush()

In [14]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


# Generate input-output pairs from the dataset

In [ ]:
import torch
import numpy as np

# block_size = maximum number of previous tokens the model can use as context

# Ensure Apple Silicon GPU (MPS) is available
if not torch.backends.mps.is_available():
    raise RuntimeError("Apple Silicon GPU (MPS) is not available.")

device = "mps"
device_type = "mps"


def get_batch(split):

    # Load the required dataset (training or validation)
    # Reloading memmap each batch avoids a known memory leak
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')

    # Randomly choose starting positions so the model learns
    # from different parts of the dataset instead of the same order
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Create input sequences (context) of length = block_size
    x = torch.stack([
        torch.from_numpy((data[i:i + block_size]).astype(np.int64))
        for i in ix
    ])

    # Create target sequences by shifting input by one token
    # This teaches the model to predict the next token
    y = torch.stack([
        torch.from_numpy((data[i + 1:i + 1 + block_size]).astype(np.int64))
        for i in ix
    ])

    # Move input and target tensors to the Apple GPU for faster training
    x = x.to(device)
    y = y.to(device)

    # Return the input-output pair for one training batch
    return x, y

# Configure the SLM architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Optional, Tuple


@dataclass
class GPTConfig:
   
    block_size: int = 256           
    vocab_size: int = 50304         
    n_layer: int = 8                
    n_head: int = 6                 
    n_kv_head: int = 2             
    n_embd: int = 384               
    dropout: float = 0.05           
    bias: bool = False              
    norm_eps: float = 1e-6          
    rope_theta: float = 10000.0     
    qk_norm: bool = True           
    attn_logit_cap: float = 50.0   
    final_logit_cap: float = 30.0   
    intermediate_size: int = 0      


# RMSNorm - a lighter version of LayerNorm

class RMSNorm(nn.Module):
    # unlike LayerNorm this skips mean-subtraction and just normalizes by the
    # root mean square, so it's cheaper to compute but works just as well

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        # doing the actual math in float32 so we don't lose precision here
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self._norm(x) * self.weight

# Rotary Positional Embeddings (RoPE)


def precompute_rope_frequencies(
    head_dim: int,
    max_seq_len: int,
    theta: float = 10000.0,
    device: Optional[torch.device] = None,
) -> torch.Tensor:
    # builds a lookup table of rotation angles for every position and every
    # frequency band, so we don't have to recompute this on every forward pass
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"
    # lower frequencies rotate slower, higher ones rotate faster - standard RoPE setup
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    # position x frequency gives us the angle for each (position, freq band) pair
    freqs = torch.outer(t, freqs)
    # storing these as complex numbers makes the rotation math a lot simpler below
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def apply_rope(
    x: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> torch.Tensor:
    # rotates q/k vectors based on their position - this is how the model
    # knows relative positions without needing learned position embeddings
    B, H, T, D = x.shape
    # pair up consecutive dims and treat each pair as one complex number
    x_complex = torch.view_as_complex(x.float().reshape(B, H, T, D // 2, 2))
    freqs = freqs_cis[:T].unsqueeze(0).unsqueeze(0)
    # multiplying by e^(i*theta) is the same as rotating - that's the whole trick
    x_rotated = torch.view_as_real(x_complex * freqs).reshape(B, H, T, D)
    return x_rotated.to(x.dtype)


# KV Cache - so generation doesn't redo work every step

class KVCache:
    # without this, generating each new token would mean re-running attention
    # over the whole sequence again. with the cache we just append and reuse.

    def __init__(self):
        self.k: Optional[torch.Tensor] = None
        self.v: Optional[torch.Tensor] = None

    def update(self, k: torch.Tensor, v: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # first call just stores it, every call after that appends along the seq dim
        if self.k is None:
            self.k = k
            self.v = v
        else:
            self.k = torch.cat([self.k, k], dim=2)
            self.v = torch.cat([self.v, v], dim=2)
        return self.k, self.v

    @property
    def seq_len(self) -> int:
        return 0 if self.k is None else self.k.size(2)

    def reset(self):
        self.k = None
        self.v = None


# Grouped Query Attention with RoPE + QK-Norm


class CausalSelfAttention(nn.Module):
    # standard causal self-attention but with a few upgrades:
    # - GQA: fewer k/v heads than q heads, saves memory and compute
    # - RoPE for position info instead of learned embeddings
    # - QK-norm so attention scores don't explode during training
    # - logit soft-cap as an extra safety net on top of that

    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.n_kv_head == 0

        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_rep = config.n_head // config.n_kv_head  # how many q heads share one kv head
        self.head_dim = config.n_embd // config.n_head
        self.n_embd = config.n_embd
        self.attn_logit_cap = config.attn_logit_cap

        # keeping q/k/v as separate projections instead of one fused matrix,
        # makes it easier to give k and v a smaller size for GQA
        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=config.bias)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=config.bias)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=config.bias)
        self.o_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.qk_norm = config.qk_norm
        if self.qk_norm:
            self.q_norm = RMSNorm(self.head_dim, eps=config.norm_eps)
            self.k_norm = RMSNorm(self.head_dim, eps=config.norm_eps)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # use the fast fused kernel when we can, fall back to manual attention otherwise
        self.use_flash = hasattr(F, 'scaled_dot_product_attention')

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        # since we have fewer k/v heads than q heads, we duplicate each kv head
        # n_rep times so the shapes line up for the attention matmul
        if self.n_rep == 1:
            return x
        B, H, T, D = x.shape
        x = x.unsqueeze(2).expand(B, H, self.n_rep, T, D)
        return x.reshape(B, H * self.n_rep, T, D)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        kv_cache: Optional[KVCache] = None,
    ) -> torch.Tensor:
        B, T, C = x.size()

        # project input into queries, keys and values
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        # normalize q/k before rope - this is what gemma 2 does and it helps stability
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # inject position info via rotation
        q = apply_rope(q, freqs_cis)
        k = apply_rope(k, freqs_cis)

        # during generation we reuse past keys/values instead of recomputing them
        if kv_cache is not None:
            k, v = kv_cache.update(k, v)

        # blow k/v back up to match the number of q heads
        k = self._repeat_kv(k)
        v = self._repeat_kv(v)

        if self.use_flash and kv_cache is None and self.attn_logit_cap == 0:
            # flash attention path - fastest option when we don't need caching or capping
            y = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
            )
        else:
            # manual path, needed whenever we're using kv cache or logit capping
            scale = 1.0 / math.sqrt(self.head_dim)
            att = (q @ k.transpose(-2, -1)) * scale

            # soft-cap keeps attention scores from getting too extreme before softmax
            if self.attn_logit_cap > 0:
                att = self.attn_logit_cap * torch.tanh(att / self.attn_logit_cap)

            # build the causal mask so each token can only look at itself and earlier tokens
            T_q, T_k = q.size(2), k.size(2)
            causal_mask = torch.triu(
                torch.full((T_q, T_k), float('-inf'), device=q.device, dtype=q.dtype),
                diagonal=T_k - T_q + 1,
            )
            att = att + causal_mask.unsqueeze(0).unsqueeze(0)

            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        # merge heads back together and project out
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.o_proj(y))
        return y


# SwiGLU Feed-Forward Network


class SwiGLUFFN(nn.Module):
    # this replaces the usual "linear -> gelu -> linear" MLP block.
    # SwiGLU adds a gating path which tends to give better results for
    # roughly the same parameter budget as a plain 4x MLP

    def __init__(self, config: GPTConfig):
        super().__init__()
        if config.intermediate_size > 0:
            hidden_dim = config.intermediate_size
        else:
            # 8/3x instead of the usual 4x since swiglu has 3 weight matrices, not 2
            hidden_dim = int(8 * config.n_embd / 3)
            # round up to a multiple of 64, plays nicer with the gpu
            hidden_dim = ((hidden_dim + 63) // 64) * 64

        self.gate_proj = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        self.up_proj = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        self.down_proj = nn.Linear(hidden_dim, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # gate path decides "how much" of the up path gets through, then project back down
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

# Transformer Block

class Block(nn.Module):
    # one transformer layer: attention + ffn, each wrapped with pre-norm and
    # a residual connection so gradients can flow through easily

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = RMSNorm(config.n_embd, eps=config.norm_eps)
        self.attn = CausalSelfAttention(config)
        self.ln2 = RMSNorm(config.n_embd, eps=config.norm_eps)
        self.ffn = SwiGLUFFN(config)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        kv_cache: Optional[KVCache] = None,
    ) -> torch.Tensor:
        # normalize first, then attend/ffn, then add back to the residual stream
        x = x + self.attn(self.ln1(x), freqs_cis, kv_cache)
        x = x + self.ffn(self.ln2(x))
        return x


# GPT Model

class GPT(nn.Module):
    # puts everything together: embedding -> n transformer blocks -> final norm -> lm head

    def __init__(self, config: Optional[GPTConfig] = None):
        super().__init__()
        if config is None:
            config = GPTConfig()
        self.config = config

        head_dim = config.n_embd // config.n_head

        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=RMSNorm(config.n_embd, eps=config.norm_eps),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # tie the embedding and output weights - saves a good chunk of params
        self.transformer.wte.weight = self.lm_head.weight

        # rope angles don't get trained, so this is just a buffer not a parameter
        self.register_buffer(
            "freqs_cis",
            precompute_rope_frequencies(head_dim, config.block_size * 2, config.rope_theta),
            persistent=False,
        )

        self.apply(self._init_weights)
        # scale down residual-path weights so deeper models stay stable early in training
        for pn, p in self.named_parameters():
            if pn.endswith('o_proj.weight') or pn.endswith('down_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        # just a quick sanity check print so we know the param count on init
        n_params = sum(p.numel() for p in self.parameters())
        n_params_no_tie = n_params - self.lm_head.weight.numel()
        print(f"Nova SLM initialized: {n_params:,} parameters ({n_params_no_tie:,} non-tied)")

    def _init_weights(self, module: nn.Module):
        # standard small-std normal init, same as gpt-2/llama use
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        kv_caches: Optional[list] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, T = idx.size()
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block_size {self.config.block_size}"

        # if we're mid-generation, figure out where in the sequence we currently are
        start_pos = 0
        if kv_caches is not None and kv_caches[0].seq_len > 0:
            start_pos = kv_caches[0].seq_len

        # no learned position embeddings here, rope takes care of that later
        x = self.transformer.drop(self.transformer.wte(idx))

        # grab just the rope angles we need for these positions
        freqs_cis = self.freqs_cis[start_pos:start_pos + T]

        for i, block in enumerate(self.transformer.h):
            cache = kv_caches[i] if kv_caches is not None else None
            x = block(x, freqs_cis, cache)

        x = self.transformer.ln_f(x)

        if targets is not None:
            # training mode: need logits at every position to compute loss
            logits = self.lm_head(x)

            if self.config.final_logit_cap > 0:
                logits = self.config.final_logit_cap * torch.tanh(logits / self.config.final_logit_cap)

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )
            return logits, loss
        else:
            # inference mode: only the last token's logits actually matter, saves compute
            logits = self.lm_head(x[:, [-1], :])

            if self.config.final_logit_cap > 0:
                logits = self.config.final_logit_cap * torch.tanh(logits / self.config.final_logit_cap)

            return logits, None

    def _create_kv_caches(self) -> list:
        # one empty cache per layer, gets filled in as generation proceeds
        return [KVCache() for _ in range(self.config.n_layer)]

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        top_p: Optional[float] = None,
        repetition_penalty: float = 1.0,
        eos_token_id: Optional[int] = None,
    ) -> torch.Tensor:
        # avoid a divide-by-zero if someone passes temperature=0
        temperature = max(temperature, 1e-7)

        kv_caches = self._create_kv_caches()

        # prefill step: run the whole prompt through once to warm up the kv cache
        prompt = idx
        if prompt.size(1) > self.config.block_size:
            prompt = prompt[:, -self.config.block_size:]

        logits, _ = self(prompt, kv_caches=kv_caches)

        # now generate one token at a time
        for _ in range(max_new_tokens):
            next_logits = logits[:, -1, :].clone()

            # push down probability of tokens we've already used, discourages repeating itself
            if repetition_penalty != 1.0:
                for b in range(idx.size(0)):
                    prev_tokens = idx[b].unique()
                    for token_id in prev_tokens:
                        if next_logits[b, token_id] > 0:
                            next_logits[b, token_id] /= repetition_penalty
                        else:
                            next_logits[b, token_id] *= repetition_penalty

            # higher temp = more random, lower temp = more confident/greedy
            next_logits = next_logits / temperature

            # only keep the top k candidates, mask out the rest
            if top_k is not None:
                k = min(top_k, next_logits.size(-1))
                v, _ = torch.topk(next_logits, k)
                next_logits[next_logits < v[:, [-1]]] = float('-inf')

            # nucleus sampling: keep the smallest set of tokens whose probs add up to top_p
            if top_p is not None and top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
                sorted_logits[sorted_mask] = float('-inf')
                next_logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)

            # sample the next token from the filtered distribution
            probs = F.softmax(next_logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

            # stop early if every sequence in the batch hit the eos token
            if eos_token_id is not None and (idx_next == eos_token_id).all():
                break

            # can't grow the kv cache past block_size, so stop here if we hit it
            if kv_caches[0].seq_len >= self.config.block_size:
                break

            # only need to forward the single new token now, cache handles the rest
            logits, _ = self(idx_next, kv_caches=kv_caches)

        return idx

    def get_num_params(self, non_embedding: bool = True) -> int:
        # subtract embedding params by default since they're tied with lm_head anyway
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wte.weight.numel()
        return n_params

In [ ]:
config = GPTConfig(
    vocab_size=50304,       
    block_size=256,         
    n_layer=8,              
    n_head=6,               
    n_kv_head=2,           
    n_embd=384,             
    dropout=0.05,           
    bias=False,             
)

learning_rate = 3e-4        
min_lr = 1e-5               
max_iters = 5000           
warmup_steps = 200         
eval_iters = 250            
eval_batches = 200          
batch_size = 32             
block_size = config.block_size
gradient_accumulation_steps = 4  
max_grad_norm = 1.0         

device = "mps" if torch.backends.mps.is_available() else "cpu"
device_type = "mps" if device == "mps" else "cpu"
dtype = "bfloat16"
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype) if device_type != "cpu" else nullcontext()
    

device = torch.device("mps")
model = GPT(config).to(device)

In [18]:
print(next(model.parameters()).device)

mps:0


In [19]:
print(torch.backends.mps.is_available())

True


#  Define the loss function

In [20]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

# Define SLM Training Configuration 

In [21]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 3e-4 # standard for small GPT models
max_iters = 5000 # reduced: with grad_accum=4, we get 1250 optimizer steps in ~4x less wall time
warmup_steps = 200 # ~4% of max_iters
min_lr = 1e-5 # must be LOWER than learning_rate for cosine decay to work
eval_iters = 250 # evaluate more frequently
batch_size = 32 # micro-batch size
block_size = 128 # context window

gradient_accumulation_steps = 4 # effective batch = 32*4 = 128 (sensible for ~29M params)

device = "mps"
device_type = "mps"
# note: float16 data type will automatically use a GradScaler
assert torch.backends.mps.is_available(), "MPS GPU not available on this machine"
dtype = "bfloat16"
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

# Enable mixed precision — this was previously nullcontext(), meaning bfloat16 was never actually used
ctx = torch.amp.autocast(device_type="mps", dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

# Define SLM Training Configuration Part 2

In [22]:
import torch
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

##PUT IN WEIGHT DECAY, CHANGED BETA2 to 0.95
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9) #weight decay for regularization

scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps) #Implement linear warmup
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr) #Implement lr decay
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps]) #Switching from warmup to decay


scaler = torch.amp.GradScaler("mps", enabled=(dtype == 'float16'))

In [23]:
X, y = get_batch("train")
X, y = X.to(device), y.to(device)

print(X.device)
print(y.device)

mps:0
mps:0


In [24]:
import torch

print(torch.__version__)
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

2.13.0
True
True


# Pre-train the SLM

In [ ]:
best_val_loss = float("inf")
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []


model = model.to(device)


for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"The current learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    
    X, y = get_batch("train")
    X, y = X.to(device), y.to(device)

    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step() # step scheduler per optimizer step, not per micro-batch

In [25]:
from pathlib import Path

print(Path("best_model_params.pt").exists())

True


In [26]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [5]:
model.load_state_dict(torch.load("best_model_params.pt", map_location=device))
model.eval()

NameError: name 'model' is not defined

# Plot the SLM Loss Function

In [28]:
import matplotlib.pyplot as plt
train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()



NameError: name 'train_loss_list' is not defined

# Perform inference using the trained SLM

In [ ]:
# Load the best model
model = GPT(config)  # re-create the model with same config
device = "mps" if torch.backends.mps.is_available() else "cpu"
best_model_params_path = "best_model_params.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device)))
model = model.to(device)
model.eval()

# Generate text
import tiktoken
enc = tiktoken.get_encoding("gpt2")

prompt = "A dog in a boat"
input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=200, temperature=0.8, top_k=40)

print(enc.decode(output[0].tolist()))


In [38]:
model.load_state_dict(torch.load("best_model_params.pt", map_location=device))
model = model.to(device)

In [40]:
losses = estimate_loss(model)
print("Train loss:", losses['train'].item())
print("Val loss:", losses['val'].item())

Train loss: 2.422661542892456
Val loss: 2.4321177005767822


In [45]:
import math
print("Perplexity:", math.exp(2.4))

Perplexity: 11.023176380641601


In [42]:
model = GPT(config)
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.load_state_dict(torch.load("best_model_params.pt", map_location=torch.device(device)))
model = model.to(device)
model.eval()

import tiktoken
enc = tiktoken.get_encoding("gpt2")

prompts = [
    "There was a little girl named",
    "Once upon a time, a dog",
    "The sun was shining and",
    "Tom and Lily went to the",
]

for p in prompts:
    input_ids = torch.tensor([enc.encode(p)], dtype=torch.long, device=device)
    out = model.generate(input_ids, max_new_tokens=100)
    print(enc.decode(out[0].tolist()))
    print("-" * 50)

There was a little girl named Lily. She had a big hat that she loved to ride it around in. One day, Lily went outside to playtime and she saw a serious task. "Let's go!" said Lily. "No, I don't want to get safe."

Lily's mom said, "But we have to be careful," said Lily. "Okay, we do something else." She stubborn to do what Lily said. She said, "It's not nice, Lily. The tornado is
--------------------------------------------------
Once upon a time, a dog named Spot went for a walk in the park. Spot was a very tight and she loved to go outside.

They would pretend he was more excited and of going out playing in the grass. One day, Spot came into the park and he saw a boy. The boy wanted to sing so much. He thought hard would be like to come with him.

"No, Spot," said Ralph. "You can pray, honey. But I can fly."

Spot did not
--------------------------------------------------
The sun was shining and the sun was shining brightly at night. Lily smiled and knew what her mom had was picked u

In [43]:
sum(p.numel() for p in model.parameters())

29995392